# fase 6: treinar e comparar modelos

In [ ]:
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)

df = pd.read_csv("../data/processed/hourly_features.csv")
df.shape

## 1. split e features (reaproveitado da fase 5)

mesmo corte por hour_utc, mesmas colunas: jan-mai treino, junho teste.

In [ ]:
hour_utc = pd.to_datetime(df["hour_utc"], utc=True)
cutoff = pd.Timestamp("2026-06-01", tz="UTC")
train_mask = hour_utc < cutoff
test_mask = hour_utc >= cutoff

FEATURE_COLUMNS = [
    "hour_of_day", "day_of_week", "is_weekend",
    "temp", "prcp", "rhum", "wspd",
    "capacity", "lat", "lon",
    "station_id",
]
TARGET_COLUMN = "is_empty"

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

## 2. duas versoes do preprocessor

regressao logistica e sensivel a escala (feature com range maior domina o coeficiente); arvore e indiferente a escala, ela so faz cortes por limiar em cada variavel. por isso: uma versao com StandardScaler no ramo numerico pra regressao logistica, outra com passthrough pra arvore.

In [ ]:
numeric_features = ["hour_of_day", "day_of_week", "is_weekend", "temp", "prcp", "wspd", "capacity", "lat", "lon"]
rhum_feature = ["rhum"]
categorical_features = ["station_id"]

# passthrough no ramo numerico: pra random forest, escala nao muda nada, entao nao faz sentido pagar o custo
preprocessor_unscaled = ColumnTransformer(transformers=[
    ("numeric", "passthrough", numeric_features),
    ("rhum", SimpleImputer(strategy="median"), rhum_feature),
    ("station", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

# StandardScaler no ramo numerico: pra regressao logistica, feature em escala maior (ex.: capacity ~30) dominaria o coeficiente sobre uma em escala menor (ex.: is_weekend 0/1)
preprocessor_scaled = ColumnTransformer(transformers=[
    ("numeric", StandardScaler(), numeric_features),
    ("rhum", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), rhum_feature),
    ("station", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

## 3. tres modelos

dummy e random forest usam o preprocessor sem escala; regressao logistica usa o com escala. os tres com class_weight="balanced" (menos o dummy, que ignora as features).

In [ ]:
models = {
    "dummy": Pipeline([
        ("preprocessor", preprocessor_unscaled),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]),
    "logistic_regression": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]),
    "random_forest": Pipeline([
        ("preprocessor", preprocessor_unscaled),
        ("classifier", RandomForestClassifier(
            n_estimators=200, min_samples_leaf=5,
            class_weight="balanced", random_state=42, n_jobs=-1,
        )),
    ]),
}

## 4. metricas no teste (junho)

f1 e a metrica de decisao. accuracy entra so pra mostrar que engana num alvo desbalanceado (~28% positivo).

In [ ]:
def evaluate_model(pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),  # dummy nunca preve a classe 1, precision vira 0/0
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "average_precision": average_precision_score(y_test, y_proba),
    }


results = []
fitted_models = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    metrics = evaluate_model(pipeline, X_test, y_test)
    metrics["model"] = name
    results.append(metrics)
    fitted_models[name] = pipeline

results_df = pd.DataFrame(results).set_index("model")
results_df = results_df[["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]]
results_df.round(4)

## 5. melhor modelo (por f1) e matriz de confusao

In [ ]:
best_model_name = results_df["f1"].idxmax()
best_model = fitted_models[best_model_name]
print("melhor modelo:", best_model_name)

y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
cm_df = pd.DataFrame(
    cm,
    index=["real: tem bike (0)", "real: vazia (1)"],
    columns=["previsto: tem bike (0)", "previsto: vazia (1)"],
)
cm_df

In [ ]:
joblib.dump(best_model, "../models/model.pkl")
print("modelo salvo em models/model.pkl:", best_model_name)

## achados

**modelo vencedor: random_forest.** f1 0.6452 contra 0.5509 da regressao logistica (+0.094, ~17% relativo) e 0.0000 do dummy. vence tambem em todas as outras metricas: roc_auc 0.8393 (logistica 0.7507), average_precision 0.6667 (logistica 0.5313), accuracy 0.7775 (logistica 0.6699).

**accuracy engana.** o dummy bate 72.15% de accuracy sem nunca prever uma estacao vazia (recall=0, f1=0). accuracy alta com f1 zero e a assinatura classica de dataset desbalanceado (~28% positivo) -- por isso a decisao usa f1, nao accuracy.

## matriz de confusao (random_forest)

|                    | previsto: tem bike | previsto: vazia |
|--------------------|--------------------|------------------|
| real: tem bike (0) | 46910              | 11938            |
| real: vazia (1)    | 6212               | 16506            |

recall 72.66%: o modelo pega quase 3 em cada 4 horas realmente vazias. os 6212 falsos negativos (27.34% das horas vazias) sao o erro mais caro no contexto do produto -- pessoa confia no app, vai ate a estacao, e nao tem bike.

precision 58.03%: dos avisos de "estacao vazia", 42% sao falso alarme (11938 falsos positivos) -- pessoa desiste de ir sem precisar.

## vale ajustar o limiar?

sim, provavelmente. o custo descrito e assimetrico: falso negativo (pessoa vai e nao tem bike, viagem perdida) e pior que falso positivo (pessoa desiste a toa, sem gastar o deslocamento). isso pesa a favor de baixar o limiar de decisao abaixo de 0.5, trocando mais falsos positivos por menos falsos negativos.

o roc_auc de 0.8393 mostra que o modelo separa bem as duas classes, entao ha espaco real pra mover o limiar sem destruir a precisao. nao foi feito nesta fase -- fica registrado como proximo passo, escolhendo o ponto de operacao numa curva precision-recall em vez do 0.5 padrao.